### Data reading in pyspark

In [0]:
# 1. Spark Session is pre-configured in Databricks as 'spark'

# 2. Let's create some dummy data
data = [("James", "Sales", 3000), 
        ("Michael", "Sales", 4600), 
        ("Robert", "Sales", 4100), 
        ("Maria", "Finance", 3000)]
columns = ["EmployeeName", "Department", "Salary"]

In [0]:
# 3. Create a PySpark DataFrame
df = spark.createDataFrame(data=data, schema=columns)

In [0]:
# --- TRANSFORMATION (Lazy) ---
# This will execute instantly because Spark is just taking notes.
high_earners_df = df.filter(df.Salary > 3500)

In [0]:
# --- ACTION (Eager) ---
# This triggers the computation across the cluster.
high_earners_df.show()

+------------+----------+------+
|EmployeeName|Department|Salary|
+------------+----------+------+
|     Michael|     Sales|  4600|
|      Robert|     Sales|  4100|
+------------+----------+------+



## Pandas to Pyspark syntax hands on testing using sample Inventory mouse hardware dataset

In [0]:
import pyspark.sql.functions as F

In [0]:
# 1. Setup: Creating the DataFrame
inventory_data = [
    ("Logitech G304", "Omron", 10.50, 50),
    ("Logitech G304", "Kailh", 12.00, 15),
    ("Razer Viper", "Kailh", 15.20, 0),
    ("SteelSeries", "Huano", 9.80, 120),
    ("Custom Build", "Huano", 11.50, 45)
]
inv_data_cols = ["PeripheralModel", "SwitchBrand", "UnitCost", "StockCount"]

In [0]:
# Reading the inv data
df_inv = spark.createDataFrame(inventory_data, inv_data_cols)

In [0]:
#Filters out any rows where the StockCount is 0.
df_stock_inv = df_inv.filter(F.col("StockCount") > 0)

In [0]:
#Creates a new column called RestockNeeded that evaluates to True if StockCount is less than 30, and False otherwise. (Hint: Look into F.when().otherwise() for conditional logic).
df_stock_inv = df_stock_inv.withColumn("RestockNeeded",F.col("StockCount") < 30)

In [0]:
#Selects only the PeripheralModel, SwitchBrand, and your new RestockNeeded column.
df_inv_final = df_stock_inv.select("PeripheralModel","SwitchBrand","RestockNeeded")

In [0]:
df_inv_final.show()

+---------------+-----------+-------------+
|PeripheralModel|SwitchBrand|RestockNeeded|
+---------------+-----------+-------------+
|  Logitech G304|      Omron|        false|
|  Logitech G304|      Kailh|         true|
|    SteelSeries|      Huano|        false|
|   Custom Build|      Huano|        false|
+---------------+-----------+-------------+



In [0]:
#Displays the final result.
display(df_inv_final)

PeripheralModel,SwitchBrand,RestockNeeded
Logitech G304,Omron,false
Logitech G304,Kailh,true
SteelSeries,Huano,false
Custom Build,Huano,false


In [0]:
# chained clean readable pipeline
df_chained_final_inv = (
    df_inv
    # .filter(F.col("StockCount")>0)
    .withColumn("RestockNeeded", F.when(F.col("StockCount") < 15, "Yes").when(F.col("StockCount") > 15, "No").otherwise("Soon"))
    .select("PeripheralModel", "SwitchBrand", "StockCount", "RestockNeeded")
)

In [0]:
display(df_chained_final_inv)

PeripheralModel,SwitchBrand,StockCount,RestockNeeded
Logitech G304,Omron,50,No
Logitech G304,Kailh,15,Soon
Razer Viper,Kailh,0,Yes
SteelSeries,Huano,120,No
Custom Build,Huano,45,No


##Aggregations & Grouping

Your Mission:
Write a single, chained PySpark query that does the following:

- Filter: Keep only the invoices where the Status is "Approved".

- Group: Group the remaining data by VendorName.

- Aggregate: Calculate two things for each vendor:

    The total sum of their approved invoices. Name this new column TotalApprovedAmount. (Hint: F.sum())

    The total count of approved invoices they have. Name this new column InvoiceCount. (Hint: F.count())

- Sort: Order the final table by TotalApprovedAmount in descending order so the vendor with the highest total is at the top. (Hint: .orderBy(F.col("TotalApprovedAmount").desc()))

- Action: Save it to a variable called df_vendor_summary and use display() to show the result.

In [0]:
import pyspark.sql.functions as F

# 1. Setup AP Data
ap_data = [
    ("INV-001", "IBM", "NA", 4500.50, "Approved"),
    ("INV-002", "Dow Chemical", "EMEIA", 12500.00, "Pending"),
    ("INV-003", "IBM", "NA", 2100.00, "Approved"),
    ("INV-004", "PepsiCo", "APAC", 890.00, "Rejected"),
    ("INV-005", "Dow Chemical", "EMEIA", 5400.75, "Approved"),
    ("INV-006", "IBM", "APAC", 7500.00, "Approved")
]
ap_columns = ["InvoiceID", "VendorName", "Region", "InvoiceAmount", "Status"]

df_ap = spark.createDataFrame(ap_data, ap_columns)

In [0]:
df_ap_final = (
    df_ap
    .filter(F.col("Status") == "Approved")
    .groupBy("VendorName")
    .agg(
        F.sum("InvoiceAmount").alias("TotalApprovedAmount"),
        F.count("InvoiceID").alias("InvoiceCount")
        )
    .orderBy(F.col("TotalApprovedAmount").desc())
)

In [0]:
display(df_ap_final)

VendorName,TotalApprovedAmount,InvoiceCount
IBM,14100.5,3
Dow Chemical,5400.75,1
